## 1 · Imports

Lädt alle externen Bibliotheken, die im Notebook verwendet werden.

| Gruppe | Bibliothek | Zweck |
|---|---|---|
| Standardbibliothek | `pathlib.Path` | Plattformunabhängige Dateipfade |
| Standardbibliothek | `time`, `logging` | Wartezeiten zwischen API-Aufrufen, strukturierte Logs |
| Datenverarbeitung | `pandas` | Tabellen (DataFrames), Parquet lesen/schreiben (Parquet-Backend `pyarrow` muss installiert, aber nicht importiert sein) |
| HTTP & API | `requests` | HTTP-Anfragen an die öffentliche Binance REST API (kein Auth nötig) |

In [ ]:
# ── Standardbibliotheken ──────────────────────────────────────
from pathlib import Path
import sys
import time
import logging

# ── Datenverarbeitung ─────────────────────────────────────────
import pandas as pd

# ── HTTP & API ────────────────────────────────────────────────
import requests

## 2 · Logging-Konfiguration

Richtet das Python-Logging ein, das alle Statusmeldungen während der Datenbeschaffung ausgibt.

Das Format `line: %(lineno)-4s %(levelname)-8s %(message)s` zeigt bei jeder Meldung:
- die **Zeilennummer** im Code (hilfreich zur Fehlersuche)
- das **Level** (`INFO` = normale Information, `WARNING` = unvollständige Daten oder übersprungene Symbole)
- die eigentliche **Nachricht**

`force=True` stellt sicher, dass die Konfiguration auch bei erneutem Ausführen der Zelle überschrieben wird. Die Variable `log` wird im gesamten Notebook als einheitliche Schnittstelle für alle Ausgaben verwendet.

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="line: %(lineno)-4s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
log = logging.getLogger(__name__)


## 3 · Pfade & Konfiguration

Alle Projektverzeichnisse (`PROJECT_ROOT`, `DATA_DIR`, `RAW_DIR`, `FETCHED_DIR`,
`PROCESSED_DIR`) sowie `ASSETS_PATH` und `OHLCV_DIR` kommen aus `notebooks/config.py`
(gemeinsam mit `01_assets.ipynb`/`01_baskets.ipynb`), damit das Notebook auf jedem Rechner
ohne Anpassung funktioniert und alle drei Notebooks garantiert auf dieselben Pfade zeigen.

**Wichtige Pfade:**
- `ASSETS_PATH`  Fixe Asset-Liste (aus `01_assets.ipynb`, inkl. `binance_symbol`-Spalte)
- `ASSET_OHLCV_5M_DIR`  Zielordner für die OHLCV-Parquet-Dateien (lokaler Alias für `OHLCV_DIR`)

**API-Parameter (notebook-spezifisch, bleiben lokal):**
- `INTERVAL = "5m"`  Jede Kerze deckt 5 Minuten ab → 288 Kerzen pro Tag
- `BINANCE_KLINE_LIMIT = 1000`  Maximale Kerzen pro API-Anfrage (Binance-Limit)
- `SLEEP_BETWEEN_REQUESTS`  Pause zwischen Anfragen, um Rate-Limits zu vermeiden
- `OHLCV_START_DATE` / `OHLCV_END_DATE`  Eigenständiges 2-Jahres-Fenster für die OHLCV-Beschaffung, unabhängig von `asset_daily`'s tagesvariabler Eligibility — die OHLCV-Universe ist jetzt eine fixe Asset-Liste über den ganzen Zeitraum.

`controlpath()` prüft beim Start, ob alle erwarteten Dateien und Ordner vorhanden sind, und gibt deren Inhalt aus.

In [ ]:
# ── Shared paths — from config.py (also used by 01_assets.ipynb / 01_baskets.ipynb) ──
sys.path.insert(0, str(Path("..").resolve()))
from config import (
    PROJECT_ROOT, DATA_DIR, RAW_DIR, FETCHED_DIR, PROCESSED_DIR,
    ASSETS_PATH, OHLCV_DIR,
)

ASSET_OHLCV_5M_DIR = OHLCV_DIR  # local alias, keeps the name used throughout this notebook

# Fetch window. Matches 01_assets.ipynb's configured START_DATE/END_DATE exactly.
# END_DATE is kept 2 days behind "today" as a safety margin against UTC/local-timezone
# edge cases and partially-elapsed days (Binance itself does not reject or flag
# incomplete-day requests - it just silently returns fewer candles - so this margin
# avoids ever asking for a day that might not be fully closed yet in UTC).
# Already-complete months (e.g. 2026-06) are skipped automatically on re-run, and the
# window can be safely extended forward later at zero cost once more days have passed.
OHLCV_START_DATE = "2024-07-23"
OHLCV_END_DATE = "2026-07-22"  # inclusive

INTERVAL               = "5m"
BINANCE_KLINE_LIMIT = 1000

def controlpath():
    log.info(f"Projektverzeichnis:            {PROJECT_ROOT}")
    log.info(f"Datenverzeichnis:              {DATA_DIR}")
    log.info(f"Rohdatenverzeichnis:           {RAW_DIR}")
    log.info(f"Gefetchte Daten:               {FETCHED_DIR}")
    log.info(f"Verarbeitete Daten:            {PROCESSED_DIR}")
    print()

    checks = {
        "Assets-Datei":              ASSETS_PATH,
        "Asset-OHLCV-5M-Verzeichnis": ASSET_OHLCV_5M_DIR,
    }

    for name, path in checks.items():
        exists = path.exists()
        symbol = "✅" if exists else "❌ FEHLT"
        log.info(f"  {symbol}  {name}: {path}")

        # Falls Ordner → Inhalt anzeigen
        if exists and path.is_dir():
            files = sorted(path.glob("*.parquet"))
            if files:
                for f in files:
                    log.info(f"       📄 {f.name}")
            else:
                log.info("       ⚠️  Ordner ist leer — keine .parquet Dateien!")


controlpath()

## 4 · OHLCV-Datenbeschaffung

Dieses Block enthält die gesamte Logik zum Herunterladen der Kerzendaten von Binance.

### `load_valid_binance_symbols()`
Fragt einmalig die Binance-API nach allen aktiven Handelssymbolen (`/api/v3/exchangeInfo`). Dient hier nur noch als defensive Zweitprüfung — die eigentliche Symbol-Auflösung passiert bereits robust in `01_assets.ipynb` über CoinGecko's Ticker-Daten.

### `load_fixed_asset_universe()`
Liest `assets.parquet` und filtert auf Assets mit aufgelöstem `binance_symbol`. Anders als früher ist diese Liste **fix für den ganzen Zeitraum** — es gibt kein tägliches Dictionary mehr, weil die Asset-Auswahl nicht mehr täglich neu bestimmt wird.

### `json_to_parquet()`
Speichert die heruntergeladenen Rohdaten als Parquet-Datei. Dabei werden die Daten nach Jahr und Monat aufgeteilt (`ohlcv_2026-01.parquet`, `ohlcv_2026-02.parquet`, …). Bereits vorhandene Einträge werden mit `drop_duplicates` nicht doppelt gespeichert. Die `timestamp`-Spalte wird beim Schreiben (und beim Einlesen bestehender Dateien vor dem Merge) explizit auf `TIMESTAMP_DTYPE` normalisiert — pandas ≥2.0 kann sonst je nach Umgebung/Version unterschiedliche `datetime64`-Auflösungen (`ns` vs. `ms`) über verschiedene Monatsdateien hinweg erzeugen, was Abschnitt 6's Schema-Check zurecht als Fehler meldet.

### `fetch_ohlcv_for_range()`  Hauptfunktion
Steuert den gesamten Ablauf:
1. Lädt die fixe Asset-Universe und gültige Binance-Symbole
2. Iteriert pro **Monat** (nicht pro Tag) über `OHLCV_START_DATE`…`OHLCV_END_DATE`
3. Prüft für jeden Monat welche Tage bereits vollständig gespeichert sind
    Ein Tag gilt als vollständig wenn der **Median** aller Ticker ≥ 270 Kerzen hat
4. Fetcht nur die fehlenden Tage/Ticker von Binance
5. Speichert die Ergebnisse via `json_to_parquet()`

**Steuerungsvariable:**

| `FORCE_REFETCH` | Verhalten |
|---|---|
| `False` | Normalbetrieb  vollständige Monate werden sofort übersprungen |
| `True` | Reparatur-Modus  jeden Ticker pro Tag einzeln prüfen und nur fehlende nachfetchen |

Quelle: `docs/uml/ohlcv_UML.drawio` (mit draw.io/diagrams.net öffnen, PNG-Export für die
Anzeige unten aktualisieren).

![UML](../../docs/uml/ohlcv_UML.png)

In [ ]:
# =========================
# Konfiguration
# =========================
BASE_URL               = "https://data-api.binance.vision"
SLEEP_BETWEEN_REQUESTS = 0.05

# False  →  nur fehlende/unvollständige Tage fetchen (Normalbetrieb)
# True   →  jeden Ticker pro Tag prüfen und fehlende nachfetchen (Reparatur-Modus)
FORCE_REFETCH = False

# datetime64 resolution all parquet files are normalized to on write. Pinning this
# explicitly avoids schema drift across pandas/pyarrow versions or machines (pandas >=2.0
# preserves the unit given to pd.to_datetime(unit=...) instead of always upcasting to ns,
# so files written by different environments can otherwise end up with different units).
TIMESTAMP_DTYPE = "datetime64[ms, UTC]"

# =========================
# Public REST
# =========================
SESSION = requests.Session()

def load_valid_binance_symbols() -> set:
    """Einmalig alle gültigen Spot-Symbole von Binance laden. Dient hier nur noch als
    defensive Prüfung (z.B. seit der Symbol-Auflösung delistete Paare) — die
    Symbol-Auflösung selbst kommt bereits robust aus assets.parquet (binance_symbol)."""
    info = SESSION.get(f"{BASE_URL}/api/v3/exchangeInfo", timeout=20).json()
    return {s["symbol"] for s in info["symbols"] if s["status"] == "TRADING"}

VALID_BINANCE_SYMBOLS: set = set()   # wird in fetch_ohlcv_for_range befüllt

def public_get(path: str, params: dict | None = None):
    url = f"{BASE_URL}{path}"
    r = SESSION.get(url, params=params, timeout=20)
    if r.status_code >= 400:
        raise RuntimeError(f"Binance Fehler {r.status_code}: {r.text}")
    return r.json()


# =========================
# Parquet Speicherung
# =========================
def json_to_parquet(data: list[dict]) -> None:
    ASSET_OHLCV_5M_DIR.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame(data)
    df["timestamp"] = df["timestamp"].dt.tz_convert("UTC").astype(TIMESTAMP_DTYPE)
    df["_year"]     = df["timestamp"].dt.year
    df["_month"]    = df["timestamp"].dt.month

    for (year, month), month_df in df.groupby(["_year", "_month"]):
        month_df = month_df.drop(columns=["_year", "_month"])
        out      = ASSET_OHLCV_5M_DIR / f"ohlcv_{year:04d}-{month:02d}.parquet"

        if out.exists():
            existing_df = pd.read_parquet(out)
            existing_df["timestamp"] = existing_df["timestamp"].astype(TIMESTAMP_DTYPE)
            df_all = (
                pd.concat([existing_df, month_df], ignore_index=True)
                .drop_duplicates(subset=["timestamp", "market_symbol"])
                .sort_values(["market_symbol", "timestamp"])
                .reset_index(drop=True)
            )
            log.info(f"  [PARQUET] Aktualisiert: {out.name}  →  {len(df_all):,} Zeilen")
        else:
            df_all = (
                month_df
                .sort_values(["market_symbol", "timestamp"])
                .reset_index(drop=True)
            )
            log.info(f"  [PARQUET] Erstellt: {out.name}  →  {len(df_all):,} Zeilen")

        df_all.to_parquet(out, index=False)


# =========================
# Fixe Asset-Universe laden
# =========================
def load_fixed_asset_universe() -> pd.DataFrame:
    """Liest nur assets.parquet (nicht mehr asset_daily) und liefert eine flache,
    fixe Liste — die OHLCV-Universe ändert sich nicht mehr täglich, sondern jedes
    Asset bekommt die volle Historie über den ganzen Zeitraum."""
    log.info("─" * 60)
    log.info(" Fixe Asset-Universe laden")
    log.info("─" * 60)
    df_assets = pd.read_parquet(ASSETS_PATH)
    log.info(f"  → {len(df_assets)} Assets geladen")

    fixed_df = df_assets.loc[df_assets["binance_symbol"].notna()].copy()
    fixed_df = fixed_df.rename(columns={"binance_symbol": "market_symbol"})
    log.info(f"  → {len(fixed_df)} Assets mit aufgelöstem Binance-Symbol")

    dropped = df_assets.loc[df_assets["binance_symbol"].isna(), ["coingecko_id", "symbol", "name"]]
    if not dropped.empty:
        log.warning(f"  ⚠️  {len(dropped)} Assets ohne Binance-Symbol — werden übersprungen: {dropped['symbol'].tolist()}")

    return fixed_df[["asset_id", "coingecko_id", "symbol", "name", "market_symbol"]].reset_index(drop=True)


# =========================
# Hauptfunktion
# =========================
def fetch_ohlcv_for_range() -> None:
    mode = "REPARATUR" if FORCE_REFETCH else "NORMAL"
    log.info("=" * 60)
    log.info(f"  OHLCV-DATENBESCHAFFUNG GESTARTET  [{mode}]")
    log.info("=" * 60)

    fixed_universe_df = load_fixed_asset_universe()

    global VALID_BINANCE_SYMBOLS
    VALID_BINANCE_SYMBOLS = load_valid_binance_symbols()
    log.info(f"  → {len(VALID_BINANCE_SYMBOLS)} gültige Binance-Symbole geladen")

    all_symbols = fixed_universe_df["market_symbol"].tolist()
    invalid     = [s for s in all_symbols if s not in VALID_BINANCE_SYMBOLS]
    symbols     = [s for s in all_symbols if s in VALID_BINANCE_SYMBOLS]
    if invalid:
        log.warning(f"  ⚠️  {len(invalid)} Symbole aktuell nicht auf Binance gelistet — werden übersprungen: {invalid}")

    window_start = pd.Timestamp(OHLCV_START_DATE).date()
    window_end   = pd.Timestamp(OHLCV_END_DATE).date()
    month_starts = pd.date_range(
        pd.Timestamp(window_start).replace(day=1), window_end, freq="MS"
    )

    for month_start_ts in month_starts:
        year, month   = month_start_ts.year, month_start_ts.month
        month_first   = pd.Timestamp(year=year, month=month, day=1).date()
        days_in_month = month_start_ts.days_in_month
        month_last    = pd.Timestamp(year=year, month=month, day=days_in_month).date()

        all_dates = pd.date_range(max(month_first, window_start), min(month_last, window_end), freq="D").date

        out = ASSET_OHLCV_5M_DIR / f"ohlcv_{year:04d}-{month:02d}.parquet"

        # Vorhandene Daten einlesen: Kerzen pro (Tag, Ticker)
        existing_dates  = set()
        ticker_candles  = {}   # (date, market_symbol) → Kerzenanzahl
        if out.exists():
            df_ex         = pd.read_parquet(out, columns=["timestamp", "market_symbol"])
            df_ex["date"] = pd.to_datetime(df_ex["timestamp"]).dt.date
            per_ticker    = df_ex.groupby(["date", "market_symbol"]).size()
            ticker_candles = per_ticker.to_dict()
            day_median    = per_ticker.groupby("date").median()
            existing_dates = set(day_median[day_median >= 270].index)
            incomplete     = set(day_median[day_median < 270].index)
            if incomplete:
                log.warning(f"  ⚠️  Unvollständige Tage: {sorted(incomplete)}")

        missing_dates = [d for d in all_dates if d not in existing_dates]

        log.info(f"\n{'=' * 60}")
        log.info(f"  Monat {year}-{month:02d}:  {len(all_dates)} Tage im Fenster  |  {len(symbols)} Ticker")
        log.info(f"  Vorhanden ({len(existing_dates)}) / Fehlend ({len(missing_dates)}): {missing_dates}")
        log.info(f"{'=' * 60}")

        if not FORCE_REFETCH:
            # Normalbetrieb: nur komplett fehlende Tage fetchen
            if not missing_dates:
                log.info("  ✅ Alle Tage vollständig — Monat wird übersprungen")
                continue
            dates_to_process = missing_dates
        else:
            # Reparatur-Modus: alle Tage im Fenster durchgehen, pro Ticker prüfen
            log.info(f"  ⚠️  REPARATUR-MODUS: alle {len(all_dates)} Tage im Fenster werden geprüft")
            dates_to_process = list(all_dates)

        fetch_errors = []

        for current_date in dates_to_process:
            # Welche Ticker brauchen noch Daten für diesen Tag?
            if FORCE_REFETCH:
                symbols_to_fetch = [
                    s for s in symbols
                    if ticker_candles.get((current_date, s), 0) < 270
                ]
                if not symbols_to_fetch:
                    log.info(f"  [✔] {current_date} — alle Ticker vollständig")
                    continue
                log.info(f"\n  ── {current_date}: {len(symbols_to_fetch)}/{len(symbols)} Ticker fehlen ──")
            else:
                symbols_to_fetch = symbols
                log.info(f"\n  ── Fetche {current_date} ({len(symbols)} Ticker) ──")

            all_rows = []
            day_errors = []

            for market_symbol in symbols_to_fetch:
                try:
                    raw = public_get("/api/v3/klines", params={
                        "symbol":    market_symbol,
                        "interval":  INTERVAL,
                        "startTime": int(pd.Timestamp(current_date, tz="UTC").timestamp() * 1000),
                        "endTime":   int((pd.Timestamp(current_date, tz="UTC") + pd.Timedelta(days=1)).timestamp() * 1000) - 1,
                        "limit":     BINANCE_KLINE_LIMIT,
                    })
                    for row in raw:
                        all_rows.append({
                            "timestamp":     pd.to_datetime(int(row[0]), unit="ms", utc=True),
                            "market_symbol": market_symbol,
                            "open":          float(row[1]),
                            "high":          float(row[2]),
                            "low":           float(row[3]),
                            "close":         float(row[4]),
                            "volume_base":   float(row[5]),
                            "volume_quote":  float(row[7]),
                            "trade_count":   int(row[8]),
                        })
                    time.sleep(SLEEP_BETWEEN_REQUESTS)
                except Exception as e:
                    msg = f"{current_date} | {market_symbol}: {e}"
                    day_errors.append(msg)
                    fetch_errors.append(msg)

            log.info(f"  → {len(all_rows):,} Kerzen  |  {len(day_errors)} Fehler")
            for err in day_errors:
                log.warning(f"    [FEHLER] {err}")

            if all_rows:
                json_to_parquet(all_rows)
            else:
                log.info(f"  [SKIP] Keine Daten für {current_date}")

        if FORCE_REFETCH and fetch_errors:
            log.warning(f"\n  ❌ Reparatur {year}-{month:02d}: {len(fetch_errors)} Fehler gesamt:")
            for err in fetch_errors:
                log.warning(f"    {err}")

    log.info("\n" + "=" * 60)
    log.info("  OHLCV-DATENBESCHAFFUNG ABGESCHLOSSEN")
    log.info("=" * 60)


# =========================
# Ausführen
# =========================
fetch_ohlcv_for_range()


## 5 · Qualitätsprüfung  `ohlcvCheck()`

Liest alle gespeicherten OHLCV-Parquet-Dateien und gibt eine Übersichtstabelle aus.

**Spalten der Tabelle:**
- `Ticker`  Anzahl unterschiedlicher Symbole an diesem Tag
- `Kerzen`  Gesamtanzahl Kerzen (Ticker × Kerzen/Ticker)
- `Kerzen/Ticker`  Durchschnittliche Kerzen pro Symbol

**Bewertung pro Tag:**
- ✅ ≥ 274 Kerzen/Ticker  vollständig (288 = 24 h × 12 Intervalle à 5 min)
- ⚠️ 200–273 Kerzen/Ticker  teilweise vorhanden
- ❌ < 200 Kerzen/Ticker  unvollständig, sollte neu gefetcht werden

Diese Zelle hat **keinen Einfluss** auf die gespeicherten Daten  sie dient nur zur Kontrolle.

In [ ]:
# =========================
# OHLCV Schnellübersicht
# =========================
def ohlcvCheck():
    log.info("=" * 70)
    log.info("OHLCV DATEI-ÜBERSICHT")
    log.info("=" * 70)

    for file in sorted(ASSET_OHLCV_5M_DIR.glob("ohlcv_*.parquet")):
        df = pd.read_parquet(file)
        df["date"] = pd.to_datetime(df["timestamp"]).dt.date

        day_ticker  = df.groupby("date")["market_symbol"].nunique()
        day_candles = df.groupby("date").size()

        log.info(f"\n📄 {file.name}")
        log.info(f"   Tage gesamt : {df['date'].nunique()}")
        log.info(f"   Ticker Ø    : {day_ticker.mean():.0f}  (min {day_ticker.min()}, max {day_ticker.max()})")
        print()
        log.info(f"   {'Datum':<14} {'Ticker':>8} {'Kerzen':>10} {'Kerzen/Ticker':>15}")
        log.info(f"   {'-'*14} {'-'*8} {'-'*10} {'-'*15}")

        for date in sorted(day_ticker.index):
            ticker  = day_ticker[date]
            candles = day_candles[date]
            ratio   = candles / ticker if ticker > 0 else 0
            status  = "✅" if ratio >= 274 else "⚠️ " if ratio >= 200 else "❌"
            log.info(f"   {str(date):<14} {ticker:>8} {candles:>10,} {ratio:>14.0f}  {status}")

    log.info("\n" + "=" * 70)
ohlcvCheck()

## 6 · Eine grosse Tabelle über den ganzen Zeitraum

Die Monats-Partitionierung (`ohlcv_YYYY-MM.parquet`) ist nur ein Speicher-Detail.
`load_full_ohlcv_table()` lädt alle Monatsdateien, prüft, dass sie identische
Spalten/Dtypes haben (deckt Schema-Drift sofort auf), und liefert damit die
eine logische Tabelle für alle Assets über den ganzen Zeitraum.

In [ ]:
def load_full_ohlcv_table() -> pd.DataFrame:
    files = sorted(ASSET_OHLCV_5M_DIR.glob("ohlcv_*.parquet"))
    if not files:
        raise FileNotFoundError(f"No OHLCV files found in {ASSET_OHLCV_5M_DIR}")

    frames = [pd.read_parquet(f) for f in files]

    reference_schema = frames[0].dtypes
    for f, frame in zip(files, frames):
        if not frame.dtypes.equals(reference_schema):
            raise ValueError(f"Schema drift detected in {f.name}: {frame.dtypes.to_dict()} != {reference_schema.to_dict()}")

    full_df = pd.concat(frames, ignore_index=True).sort_values(["market_symbol", "timestamp"]).reset_index(drop=True)
    return full_df


full_ohlcv_df = load_full_ohlcv_table()

print(f"Total rows:        {len(full_ohlcv_df):,}")
print(f"Distinct symbols:  {full_ohlcv_df['market_symbol'].nunique()}")
print(f"Date range:        {full_ohlcv_df['timestamp'].min()}  →  {full_ohlcv_df['timestamp'].max()}")
print(f"Columns:           {full_ohlcv_df.columns.tolist()}")

## 7 · Zwischenergebnis-Inspektion

Zeigt alle Zwischenstufen der Datenpipeline im Detail an  nur zur Analyse und zum Verständnis des Datenflusses. Diese Zelle verändert keine Daten und muss nicht bei jedem Lauf ausgeführt werden.

**Dargestellte Stufen:**

| Stufe | Inhalt |
|---|---|
| `df_assets` | Alle Coins aus `assets.parquet` mit Metadaten |
| `fixed_universe_df` | Fixe Ticker-Liste (nur Assets mit aufgelöstem `binance_symbol`) |
| `raw` | Rohe Binance API-Antwort (Liste von 12-elementigen Arrays) |
| `all_rows` | Umgewandelte Dicts mit sprechenden Spaltennamen |
| `df` | Pandas DataFrame bereit zum Speichern |
| Parquet-Dateien | Finales Ergebnis mit Shape, Ticker- und Tagesanzahl |

In [ ]:
# =============================================================
# ZWISCHENERGEBNIS-INSPEKTION — alle Stufen aus dem Diagramm
# =============================================================

SEP = "=" * 65

# ── 1. df_assets ─────────────────────────────────────────────
print(f"\n{SEP}")
print("1) df_assets   (aus assets.parquet)")
print(SEP)
df_assets = pd.read_parquet(ASSETS_PATH)
print(f"  Typ    : {type(df_assets)}")
print(f"  Shape  : {df_assets.shape}  →  {df_assets.shape[0]} Assets, {df_assets.shape[1]} Spalten")
print(f"  Spalten: {df_assets.columns.tolist()}")
print(f"  Dtypes :\n{df_assets.dtypes.to_string()}")
print("\n  Erste 3 Zeilen:")
display(df_assets.head(3))

# ── 2. fixed_universe_df ─────────────────────────────────────
print(f"\n{SEP}")
print("2) fixed_universe_df   (nur Assets mit aufgelöstem binance_symbol)")
print(SEP)
_fixed = load_fixed_asset_universe()
print(f"  Typ    : {type(_fixed)}")
print(f"  Shape  : {_fixed.shape}")
print(f"  Spalten: {_fixed.columns.tolist()}")
display(_fixed.head(5))

# ── 3. raw   (ein echter Binance-API Aufruf) ─────────────────
print(f"\n{SEP}")
print("3) raw   (Binance API Antwort — JSON Liste)")
print(SEP)
_example_symbol = _fixed["market_symbol"].iloc[0]
_raw = public_get("/api/v3/klines", params={
    "symbol":    _example_symbol,
    "interval":  "5m",
    "startTime": int(pd.Timestamp(OHLCV_START_DATE, tz="UTC").timestamp() * 1000),
    "endTime":   int(pd.Timestamp(f"{OHLCV_START_DATE} 00:30", tz="UTC").timestamp() * 1000),
    "limit":     6,
})
print(f"  Symbol          : {_example_symbol}")
print(f"  Typ             : {type(_raw)}")
print(f"  Anzahl Kerzen   : {len(_raw)}")
print(f"  Elemente pro Kerze: {len(_raw[0])}")
print("\n  Erste Kerze (rohe Werte):")
_labels = ["open_time","open","high","low","close","volume","close_time",
           "quote_volume","trades","taker_buy_base","taker_buy_quote","ignore"]
for label, val in zip(_labels, _raw[0]):
    print(f"    [{_labels.index(label)}] {label:<20} = {val}")

# ── 4. all_rows   (nach Umwandlung in Dicts) ─────────────────
print(f"\n{SEP}")
print("4) all_rows   (Liste von Dictionaries)")
print(SEP)
_all_rows = []
for _row in _raw:
    _all_rows.append({
        "timestamp":     pd.to_datetime(int(_row[0]), unit="ms", utc=True),
        "market_symbol": _example_symbol,
        "open":          float(_row[1]),
        "high":          float(_row[2]),
        "low":           float(_row[3]),
        "close":         float(_row[4]),
        "volume_base":   float(_row[5]),
        "volume_quote":  float(_row[7]),
        "trade_count":   int(_row[8]),
    })
print(f"  Typ             : {type(_all_rows)}")
print(f"  Anzahl Einträge : {len(_all_rows)}")
print(f"  Keys eines Eintrags: {list(_all_rows[0].keys())}")
print("\n  Erster Eintrag:")
for k, v in _all_rows[0].items():
    print(f"    {k:<20} = {v}  ({type(v).__name__})")

# ── 5. df = pd.DataFrame(all_rows) ──────────────────────────
print(f"\n{SEP}")
print("5) df = pd.DataFrame(all_rows)   (Tages-DataFrame)")
print(SEP)
_df_day = pd.DataFrame(_all_rows)
print(f"  Typ    : {type(_df_day)}")
print(f"  Shape  : {_df_day.shape}  →  {_df_day.shape[0]} Kerzen, {_df_day.shape[1]} Spalten")
print(f"  Dtypes :\n{_df_day.dtypes.to_string()}")
print("\n  Alle Zeilen:")
display(_df_day)

# ── 6. Finale Parquet-Dateien ────────────────────────────────
print(f"\n{SEP}")
print("6) parquet files pro Monat   (Endresultat)")
print(SEP)
_ohlcv_files = sorted(ASSET_OHLCV_5M_DIR.glob("ohlcv_*.parquet"))
print(f"  Anzahl Dateien: {len(_ohlcv_files)}")
for _f in _ohlcv_files:
    _df_f = pd.read_parquet(_f)
    print(f"\n  📄 {_f.name}")
    print(f"     Shape  : {_df_f.shape}  →  {_df_f.shape[0]:,} Kerzen, {_df_f.shape[1]} Spalten")
    print(f"     Ticker : {_df_f['market_symbol'].nunique()}")
    print(f"     Tage   : {pd.to_datetime(_df_f['timestamp']).dt.date.nunique()}")
    print(f"     Von    : {_df_f['timestamp'].min()}")
    print(f"     Bis    : {_df_f['timestamp'].max()}")

print(f"\n{SEP}")
print("INSPEKTION ABGESCHLOSSEN")
print(SEP)
